# Vector Stores -- Indexing and Searching Embeddings

## What You Will Learn

Once documents are chunked, the next step in a RAG pipeline is to convert those chunks into **vector embeddings** and store them in a **vector store** for fast similarity search.

This notebook covers:

1. **What embeddings are** -- dense vectors that capture the *meaning* of text.
2. **Vector similarity** -- how cosine similarity, dot product, and L2 distance measure closeness in embedding space.
3. **Vector store options** -- comparing ChromaDB (persistent, metadata filtering) and FAISS (raw speed, billion-scale).

## Prerequisites

* Python 3.10+
* `numpy` (for the from-scratch similarity demo)
* The `agentexplorr` package installed

No embedding model or external service is needed for this notebook -- we work with hand-crafted vectors to build intuition.

## Where This Fits in the Pipeline

```
Chunks  -->  EmbeddingModel  -->  VectorStore  -->  Similarity Search
              (text -> vec)       (index vecs)      (query -> top-K results)
```

In [ ]:
import numpy as np

# --- The core idea: text becomes a vector, similar text = similar vectors ---

# Imagine a tiny 4-dimensional embedding space.
# In reality, models like all-MiniLM-L6-v2 produce 384-dimensional vectors,
# but 4D is enough to illustrate the concept.

# Two sentences about animals -- their vectors should be similar
vec_cat  = np.array([0.9, 0.1, 0.0, 0.2])   # "The cat sat on the mat"
vec_dog  = np.array([0.85, 0.15, 0.05, 0.18]) # "A dog rested on the rug"

# A sentence about finance -- its vector should be far from the animal sentences
vec_stock = np.array([0.1, 0.8, 0.7, 0.05])  # "Stock prices rose sharply"

print("Embedding vectors (simplified 4D):")
print(f"  'The cat sat on the mat'   -> {vec_cat}")
print(f"  'A dog rested on the rug'  -> {vec_dog}")
print(f"  'Stock prices rose sharply' -> {vec_stock}")

# Dot product: a raw measure of directional alignment
dot_similar = np.dot(vec_cat, vec_dog)
dot_different = np.dot(vec_cat, vec_stock)

print(f"\nDot product (cat vs dog):   {dot_similar:.4f}  (high = similar)")
print(f"Dot product (cat vs stock): {dot_different:.4f}  (low = different)")

## Vector Similarity Metrics

Vector stores need a way to measure how "close" two vectors are. The three most common metrics are:

### Cosine Similarity

Measures the **angle** between two vectors, ignoring magnitude. Range: -1 (opposite) to +1 (identical direction).

$$\text{cosine}(A, B) = \frac{A \cdot B}{\|A\| \times \|B\|}$$

This is the most popular metric for text embeddings because it focuses on *direction* (meaning) rather than *length* (which can vary with text length).

### Dot Product (Inner Product)

Simply $A \cdot B = \sum a_i \cdot b_i$. When vectors are **L2-normalized** (length = 1), the dot product equals cosine similarity. This is why `agentexplorr`'s `EmbeddingModel` normalizes by default -- it makes dot product equivalent to cosine but cheaper to compute.

### L2 (Euclidean) Distance

The straight-line distance: $\|A - B\|_2 = \sqrt{\sum (a_i - b_i)^2}$. **Lower** means more similar (unlike cosine, where higher is better). FAISS uses this by default with `IndexFlatL2`.

| Metric | Range | Similar = | Used By |
|---|---|---|---|
| Cosine Similarity | [-1, 1] | High (close to 1) | ChromaDB (default) |
| Dot Product | unbounded | High | FAISS `IndexFlatIP` |
| L2 Distance | [0, inf) | Low (close to 0) | FAISS `IndexFlatL2` |

In [ ]:
def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    """Compute cosine similarity between two vectors from scratch.

    Formula: cos(theta) = (A . B) / (||A|| * ||B||)
    """
    dot = np.dot(a, b)
    norm_a = np.linalg.norm(a)
    norm_b = np.linalg.norm(b)
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return float(dot / (norm_a * norm_b))


def l2_distance(a: np.ndarray, b: np.ndarray) -> float:
    """Compute Euclidean (L2) distance between two vectors."""
    return float(np.linalg.norm(a - b))


# --- Compare all three metrics across our example vectors ---
pairs = [
    ("cat vs dog   (similar)", vec_cat, vec_dog),
    ("cat vs stock (different)", vec_cat, vec_stock),
    ("dog vs stock (different)", vec_dog, vec_stock),
]

print(f"{'Pair':<30} {'Cosine':>8} {'Dot':>8} {'L2 Dist':>8}")
print("-" * 60)
for label, a, b in pairs:
    cos = cosine_similarity(a, b)
    dot = float(np.dot(a, b))
    l2 = l2_distance(a, b)
    print(f"{label:<30} {cos:>8.4f} {dot:>8.4f} {l2:>8.4f}")

print("\nNotice:")
print("  - Cosine similarity is HIGH for similar text (cat/dog) and LOW for different topics.")
print("  - L2 distance is LOW for similar text and HIGH for different topics.")
print("  - When vectors are normalized, cosine and dot product give the same ranking.")

# Demonstrate that normalization makes dot product = cosine similarity
norm_cat = vec_cat / np.linalg.norm(vec_cat)
norm_dog = vec_dog / np.linalg.norm(vec_dog)
print(f"\nAfter L2-normalization:")
print(f"  cosine(cat, dog) = {cosine_similarity(norm_cat, norm_dog):.6f}")
print(f"  dot(cat, dog)    = {float(np.dot(norm_cat, norm_dog)):.6f}")
print(f"  They are equal!")

## ChromaDB vs FAISS -- Choosing a Vector Store

AgentExplorr ships with two vector store backends. Each has distinct strengths:

### ChromaDB (`agentexplorr.rag.vector_stores.chroma_store`)

* **Persistence out of the box** -- data survives restarts without extra code.
* **Metadata filtering** -- SQL-like filters on chunk metadata (e.g., `{"source": "paper.pdf", "page": {"$gt": 5}}`).
* **Built-in deduplication** -- upserting the same chunk ID overwrites the previous entry.
* **Index algorithm**: HNSW (Hierarchical Navigable Small World) -- O(log N) query time.
* **Best for**: prototyping, small-to-medium datasets (< 1M chunks), applications needing metadata filters.

### FAISS (`agentexplorr.rag.vector_stores.faiss_store`)

* **Raw speed** -- optimized C++ with optional GPU acceleration. Battle-tested at Meta on billion-vector datasets.
* **Multiple index types** -- `IndexFlatL2` (exact, 100% recall), `IndexIVFFlat` (approximate, 10-100x faster), `IndexHNSWFlat`, `IndexIVFPQ` (compressed, billion-scale).
* **No external server** -- it is a library, not a service.
* **Trade-off**: FAISS only stores vectors. Text, metadata, and chunk IDs must be stored separately (AgentExplorr's `FAISSVectorStore` handles this with a parallel `_documents` list).
* **Best for**: large-scale production, latency-critical search, GPU-accelerated workloads.

### Quick Comparison

| Feature | ChromaDB | FAISS |
|---|---|---|
| Persistence | Built-in | Manual (save/load) |
| Metadata filtering | Yes | No (must filter post-search) |
| Approximate NN | HNSW only | Many index types |
| GPU support | No | Yes (`faiss-gpu`) |
| Scale sweet spot | < 1M vectors | 1M -- 1B+ vectors |
| Install | `pip install chromadb` | `pip install faiss-cpu` |

In AgentExplorr, both stores expose the same `add_documents()` and `search()` interface, so the `HybridRetriever` can query them interchangeably.

## Key Takeaways

1. **Embeddings encode meaning as vectors.** Similar text produces vectors that point in similar directions, enabling search by *meaning* rather than keyword matching.

2. **Cosine similarity is the go-to metric** for text embeddings. It ignores vector magnitude and focuses on direction, which correlates with semantic similarity.

3. **L2-normalizing vectors** makes dot product equivalent to cosine similarity. AgentExplorr's `EmbeddingModel` normalizes by default, so both ChromaDB and FAISS produce comparable scores.

4. **ChromaDB** is the best choice for getting started -- zero-config persistence, metadata filtering, and built-in deduplication. Use it for prototypes and datasets under 1M chunks.

5. **FAISS** is the best choice for scale and speed -- optimized C++ with GPU support and multiple index types. Use it when you need sub-millisecond search over millions of vectors.

6. Both stores implement the same interface (`add_documents`, `search`), so AgentExplorr's `HybridRetriever` can query them together for **higher recall through ensemble retrieval**.

## Next Steps

* **03_rag_pipeline.ipynb** -- See how the `HybridRetriever` combines results from multiple stores and re-ranks them with Reciprocal Rank Fusion (RRF).
* Try swapping ChromaDB for FAISS in your pipeline -- the interface is identical, so it is a one-line change.